# 04 · Pick 300 roofs for hand-labelling

**Run this in Google Colab.** It takes about 5 minutes and uses only DuckDB (no Spark).

What it does:
1. Reads the roof features made by notebook 03.
2. Picks **21 random roofs from each of the 14 districts** (294 roofs) + 6 extra = **300 roofs**.
3. Saves them as a **GeoPackage** file for QGIS: `MyDrive/rooftop_solar/data/labels/roofs_to_label.gpkg`

The file has only the roof outline, an id, the district, and two empty columns for you:
- `label`: type **1** = good, **0** = bad, **2** = not a building (leave empty if you cannot tell)
- `note`: optional comment

Roof size, tree cover and other features are **not** in the file on purpose, so your judgement stays independent of the model.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive and set folders
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

PROJECT = '/content/drive/MyDrive/rooftop_solar'
FEATURES = f'{PROJECT}/data/features/roof_features'
LABELS = f'{PROJECT}/data/labels'
LOCAL = '/content/work'
os.makedirs(LABELS, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

OUT_NAME = 'roofs_to_label.gpkg'
if os.path.exists(f'{LABELS}/{OUT_NAME}'):
    raise SystemExit(f'{LABELS}/{OUT_NAME} already exists. It may hold your labels, so it is not replaced. '
                     'Rename or move it first if you really want a new sample.')
print('Reading roof features from', FEATURES)

In [ ]:
# Step 2: install DuckDB
!pip install -q duckdb==2.0.0

import duckdb

con = duckdb.connect()
con.sql("INSTALL spatial; LOAD spatial;")
con.sql("SET geometry_always_xy = true;")   # coordinates are (longitude, latitude)

In [ ]:
# Step 3: pick 21 roofs per district at random (repeatable: the same 300 roofs every time)
PER_DISTRICT = 21
TOTAL = 300

con.execute(f"""
    CREATE TABLE picked AS
    WITH ranked AS (
        SELECT id, district, low_confidence, geometry,
               row_number() OVER (PARTITION BY district ORDER BY hash(id || 'labels-v1')) AS r
        FROM read_parquet('{FEATURES}/*/*.parquet', hive_partitioning = true)
    ),
    base AS (SELECT * FROM ranked WHERE r <= {PER_DISTRICT}),
    extra AS (                       -- top up to 300 with the next roofs, spread over districts
        SELECT * FROM ranked WHERE r = {PER_DISTRICT} + 1
        ORDER BY hash(id || 'labels-v1')
        LIMIT {TOTAL} - (SELECT count(*) FROM base)
    )
    SELECT * FROM base UNION ALL SELECT * FROM extra
""").fetchall()

# Number the roofs district by district, west to east, so you move around the map less
con.execute(f"""
    COPY (
        SELECT row_number() OVER (ORDER BY district, ST_X(ST_Centroid(geometry))) AS sample_no,
               id, district,
               CAST(NULL AS INTEGER) AS label,
               CAST(NULL AS VARCHAR) AS note,
               geometry
        FROM picked
    ) TO '{LOCAL}/{OUT_NAME}' WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'roofs_to_label', SRS 'EPSG:4326')
""").fetchall()

print(con.sql("""
    SELECT district, count(*) AS roofs, sum(CAST(low_confidence AS INT)) AS low_confidence_ai_shapes
    FROM picked GROUP BY district ORDER BY district
"""))
print('Total roofs picked:', con.execute('SELECT count(*) FROM picked').fetchall()[0][0])

In [ ]:
# Step 4: save the file to Google Drive
shutil.copy(f'{LOCAL}/{OUT_NAME}', f'{LABELS}/{OUT_NAME}')
print('Saved:', f'{LABELS}/{OUT_NAME}')
print('Next: download this file from Google Drive (rooftop_solar/data/labels) and open it in QGIS.')

## Done

1. In Google Drive, open `rooftop_solar/data/labels`, right-click `roofs_to_label.gpkg` → **Download**.
2. Open it in QGIS and label the roofs (Claude gave you the steps).
3. When finished, upload the saved file back to the same Drive folder.